In [ ]:
from vllm import LLM, SamplingParams
from transformers import pipeline, GenerationConfig
import pandas as pd
import numpy as np
import torch


def get_perplexity(token_list):
    """
        Calcula perplexity para una instancia única.
    """
    t = len(token_list)
    avg = (-1/t) * np.sum(token_list)
    perplexity = np.exp(avg)
    return round(perplexity, 4)

val = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl'
dataset = pd.read_json(val, lines = True)

trans_prompt = """
    Given a set of premises, the task is to parse the problem and the question into first-order logic formulars. Answer only with the translated premises.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Natural Language Premises:
    All people who regularly drink coffee are dependent on caffeine. People either regularly drink coffee or joke about being addicted to caffeine. No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    Predicates:
    Dependent(x) ::: x is a person dependent on caffeine.
    Drinks(x) ::: x regularly drinks coffee.
    Jokes(x) ::: x jokes about being addicted to caffeine.
    Unaware(x) ::: x is unaware that caffeine is a drug.
    Student(x) ::: x is a student.
    FOL Premises:
    ∀x (Drinks(x) → Dependent(x)) ::: All people who regularly drink coffee are dependent on caffeine.
    ∀x (Drinks(x) ⊕ Jokes(x)) ::: People either regularly drink coffee or joke about being addicted to caffeine.
    ∀x (Jokes(x) → ¬Unaware(x)) ::: No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. 
    (Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina)) ::: Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. 
    ¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina)) ::: If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    --------------
    
    Natural Language Premises:
    {}
    FOL Premises:
"""

infer_prompt = """
    Given a set of premises and conclusion in first order logic, your task is to determine the logical validity of the conclusion: True, False, or Uncertain. Answer only with the logical value.
    A True conclusion is one that can be obtained via a valid inference procedure from the given premises.
    A False conclusion is one that contradicts one or more premises during the inference procedure. 
    An Uncertain conclusion is neither True nor False. Meaning that there is insufficient information in the premises to infer it, but the conclusion it self doesn't contradict any premise.
    --------------
    The following example shows a set of premises and conclusions where each conclusion represents a different logical validity. You should answer similarly.
    FOL-PREMISES:
    ∀x (WorkAt(x, meta) → HighIncome(x))
    ∀x (HighIncome(x) → ¬MeansToDestination(x, bus))
    ∀x (MeansToDestination(x, bus) ⊕ MeansToDestination(x, drive))
    ∀x (HaveCar(x) → MeansToDestination(x, drive))
    ∀x (Student(x) → ¬ MeansToDestination(x, drive))
    HaveCar(james) ∨ WorkAt(james, meta)
    --------------
    FOL-CONCLUSION:
    MeansToDestination(x, drive) ∨ Student(james)
    Student(james)
    ¬HighIncome(james)

    Analysis:
    The first conclusion is True. Premise 6 states that either James has a car (in which case premise 4 gives us the conclusion) or James works at Meta (in which case premise 4 implies premise 2, which combined with premise 3 gives us the conclusion)
    The second conclusion is False. Premise 5 states that students can't have a Car as a MeansToDestination, however the first condition tells us James has such means.
    The third conclusion is Uncertain. Premise 1 is the only guarantee to have a High Income, however we can't determine that James works at Meta (Premise 6).
    ----------------------------
    FOL-PREMISES:
    {}
    --------------
    FOL-CONCLUSION:
    {}
    --------------
    ANSWER:
"""


retrans_prompt = """
    Given a single premise in first order logic, your task is to translate the premise into natural language. Answer only with the translated premise. It should be a single sentence.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Below are examples of the translation:
    PREMISES:
    ¬(PartTime(jackie) ⊕ ForbesList(jackie)) → ∃y (LessThan(y, num2) ∧ TakesCourses(x,y)) ∧ ForbesList(jackie)
    ¬In(borjMasouda, tunisia)

    NATURAL LANGUAGE:
    If Jackie either enrolls as part-time in the current semester and is listed in the Forbes 30 Under 30, or neither enrolls as part-time in the current semester nor is listed in the Forbes 30 Under 30, then Jackie takes less than two courses in the current semester and listed in the Forbes 30 Under 30.
    Borj Masouda is not in Tunisia.
    --------------    
    PREMISE:
    {}

    NATURAL LANGUAGE:
"""

In [4]:
from vllm.lora.request import LoRARequest
def vllm_generation(model_id, quant):
    """
        Generación iterativa sobre distintos modelos usando vLLM.
    """
    print(f"Iniciando con: {model_id}...")
    
    trans_prompts = [trans_prompt.format(dataset['premises'][i]) for i in range(len(dataset['premises']))]
    infer_prompts = [infer_prompt.format(dataset['premises-FOL'][i], dataset['conclusion-FOL'][i]) for i in range(len(dataset['premises-FOL']))]
    retrans_prompts = [retrans_prompt.format(dataset['conclusion-FOL'][i]) for i in range(len(dataset['conclusion-FOL']))]

    prompts = trans_prompts + infer_prompts + retrans_prompts
    sampling_params = SamplingParams(temperature = 0.2, top_p = 0.9, max_tokens = 4000, seed = 67, logprobs = 1)
    llm = LLM(
        model = model_id, 
        enable_lora = True,
        max_model_len = 6000, 
        quantization = quant, 
        enforce_eager = True, 
        gpu_memory_utilization = 0.9, 
        limit_mm_per_prompt={"image": 0, "video": 0}
    )
    outputs = llm.generate(prompts, sampling_params, lora_request=None)

    step_list = []
    perp_list = []
    i = 0
    for output in outputs:
        gen_text = output.outputs[0].text.strip(r'\n')
        step_list.append(gen_text)
        loglist = output.outputs[0].logprobs
        logit_values = [loglist[i].get(list(loglist[i].keys())[0]).logprob for i in range(len(loglist))]
        perp_list.append(get_perplexity(logit_values))
        if i % 20 == 0:
            print('Iteración: {}'.format(i))
        i += 1

        
    # To do: Pensar en otras cosas que medirle a las generaciones de los modelos.
    df_name = model_id.split('/')[1] # AHHHHHHHHHHHHHHHHHHHHHHH
    full_dataframe = pd.DataFrame({f"Full": step_list, "Perplexity": perp_list})
    full_dataframe.to_csv(f'/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/{df_name}_full.csv')
    print("=" * 60)
    print("=" * 15 +  f" Fin de: {model_id} " + "=" * 15)
    print("=" * 60)


checkpoint_list = [
    ['Qwen/Qwen3-4B-FP8', 'fp8'], # Funciona bien. DONE
    ['Qwen/Qwen3-8B-FP8', 'fp8'], # Funciona bien. DONE
    ['Qwen/Qwen3-14B-FP8', 'fp8'], # Fuinciona bien. DONE
    ['Qwen/Qwen3.5-4B', 'fp8'], # Funciona bien. DONE # Estamos haciendo pruebas con max_model_len = 7000, y max_tokens = 5000
    ['Qwen/Qwen3.5-9B', 'fp8'], # Funciona bien. DONE
    ['google/gemma-3-4b-it','fp8'], # Funciona bien. DONE
    ['google/gemma-3-12b-it','fp8'], # Funciona bien. DONE
    ['openai/gpt-oss-20b', 'mxfp4'], # Funciona bien. DONE
    ['deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', None], # Jala bien.
    ['deepseek-ai/DeepSeek-R1-0528-Qwen3-8B', None]
]

KTO_checkpoint_list = [
    ['Kurosawama/KTO_DeepSeek-R1-0528-Qwen3-8B', None],
    ['Kurosawama/KTO_Qwen3-14B', None],
    ['Kurosawama/KTO_gemma-3-12b-it', None]
]

for _ in KTO_checkpoint_list:
    vllm_generation(_[0], _[1])
#vllm_generation(checkpoint_list[0][0], checkpoint_list[0][1])

Iniciando con: Kurosawama/KTO_DeepSeek-R1-0528-Qwen3-8B...
INFO 05-15 05:46:11 [utils.py:233] non-default args: {'max_model_len': 6000, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'enable_lora': True, 'model': 'Kurosawama/KTO_DeepSeek-R1-0528-Qwen3-8B'}


ValidationError: 1 validation error for ModelConfig
  Value error, Invalid repository ID or local directory specified: 'Kurosawama/KTO_DeepSeek-R1-0528-Qwen3-8B'.
Please verify the following requirements:
1. Provide a valid Hugging Face repository ID.
2. Specify a local directory that contains a recognized configuration file.
   - For Hugging Face models: ensure the presence of a 'config.json'.
   - For Mistral models: ensure the presence of a 'params.json'.
 [type=value_error, input_value=ArgsKwargs((), {'model': ...rocessor_plugin': None}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [ ]:
# HAY QUE PASARLE TODOS LOS VALORES DE SALIDA AL MODELO AHORA AHHHHH.
trans_prompts = [trans_prompt.format(dataset['premises'][i]) for i in range(len(dataset['premises']))]
infer_prompts = [infer_prompt.format(dataset['premises-FOL'][i], dataset['conclusion-FOL'][i]) for i in range(len(dataset['premises-FOL']))]
retrans_prompts = [retrans_prompt.format(dataset['conclusion-FOL'][i]) for i in range(len(dataset['conclusion-FOL']))]
prompts = trans_prompts + infer_prompts + retrans_prompts

outputs = []
generator = pipeline("text-generation", model="Kurosawama/KTO_DeepSeek-R1-0528-Qwen3-8B", device="cuda")
for i in range(len(prompts)):
    print('-' * 20, i, '-' * 20)
    question = prompts[i]
    output = generator([{"role": "user", "content": question}], max_new_tokens=4000, temperature = 0.2, return_full_text=False)[0]
    print(output["generated_text"])
    outputs.append(output['generated_text'])
    del output
    torch.cuda.empty_cache()

print(len(outputs))
df_name = 'KTO_DeepSeek-R1-0528-Qwen3-8B'
full_dataframe = pd.DataFrame({f"Full": outputs})
full_dataframe.to_csv(f'/home/flopezp/Kurosagol/Ongoing/alignment_results/validation/{df_name}_full.csv')
del generator
torch.cuda.empty_cache()


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-------------------- 0 --------------------


...

No one who plays for Nautico is Brazilian.
-------------------- 605 --------------------


Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No one plays for a football club.

No one plays for a football club.
-------------------- 606 --------------------
False.
-------------------- 607 --------------------


Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No matter what, Ailton Silva plays for Fluminense.
-------------------- 608 --------------------
No one is a football club except for Ailton Silva.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton Silva is not a football club.

    Ailton 

In [ ]:
def evaluate_kto_models(validation, checkpoint):
    if validation:
        dataset = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines=True)
        split = 'validation'
    else:
        dataset = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
        split = 'test'
    
    trans_prompts = [trans_prompt.format(dataset['premises'][i]) for i in range(len(dataset['premises']))]
    infer_prompts = [infer_prompt.format(dataset['premises-FOL'][i], dataset['conclusion-FOL'][i]) for i in range(len(dataset['premises-FOL']))]
    retrans_prompts = [retrans_prompt.format(dataset['conclusion-FOL'][i]) for i in range(len(dataset['conclusion-FOL']))]
    prompts = trans_prompts + infer_prompts + retrans_prompts

    ds_name = checkpoint.split('/')[-1]
    outputs = []
    generator = pipeline('text-generation', model = checkpoint, device = 'cuda')
    generation_confg = GenerationConfig(
        max_new_tokens = 4000,
        temperature = 0.2,
    )
    for i in range(len(prompts)):
        if i % 20 == 0:
            print('-' * 20, i, '-' * 20)
        question = prompts[i]
        output = generator([{"role": "user", "content": question}], generation_confg = generation_confg, return_full_text=False)[0]
        #print(output["generated_text"])
        outputs.append(output['generated_text'])
        del output
        torch.cuda.empty_cache()
    
    print('Length: {}'.format(len(outputs)))
    full_dataframe = pd.DataFrame({f"Full": outputs})
    full_dataframe.to_csv(f'/home/flopezp/Kurosagol/Ongoing/alignment_results/{split}/{ds_name}_full.csv')
    del generator
    torch.cuda.empty_cache()


KTO_checkpoint_list = [
    'Kurosawama/KTO_Qwen3-14B',
    'Kurosawama/KTO_gemma-3-12b-it'
]

evaluate_kto_models(False, 'Kurosawama/KTO_DeepSeek-R1-0528-Qwen3-8B')

for elem in KTO_checkpoint_list:
    print(f'Empezando con: {elem}...')
    evaluate_kto_models(True, elem)
    evaluate_kto_models(False, elem)
    print('='*60)
    print('='*60)

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-------------------- 0 --------------------


...

Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Length: 678
Empezando con: {elem}...


adapter_config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

KeyError: 'llava'